# Image creation from audio data



# Image generation process
- Compute dB scaled mel power spectrum over 5 seconds interval.
- Use primary label for each of these intervals.
- Pad to 5 second images if we have a minimal duration.
- Consider a maximum duration for a maximum number of images created per file.
- Create images independently.

In [1]:
import os, pathlib
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from pydantic import BaseModel as ConfigBaseModel
from sklearn.preprocessing import LabelEncoder
from joblib import delayed, Parallel
import librosa
print("librosa:", librosa.__version__)
import tensorflow as tf
print("tensorflow:", tf.__version__)
import cv2
print("opencv:", cv2.__version__)
from IPython.display import Audio
from joblib import Parallel, delayed
import soundfile as sf

from google.colab import drive
drive.mount('/content/drive')

librosa: 0.10.2.post1
tensorflow: 2.18.0
opencv: 4.11.0
Mounted at /content/drive


# Config

In [22]:
from pydantic import BaseModel
from typing import Tuple

class Config(BaseModel):
    # Datos de configuración principales
    base_dir: str = "/content/drive/My Drive/"
    train_sound_dir: str = "/content/drive/My Drive/songs/"
    path_train: str = "/content/drive/My Drive/data_passerifromes/paths_cantos_passeriformes.csv"
    sample_rate: int = 32_000
    # Configuración del espectrograma
    img_size: Tuple[int, int] = (480, 480)
    seconds: int = 5
    num_offset_max: int = 24
    min_duration: float = 0.5
    n_fft: int = 2048
    n_mels: int = 480  # Corresponde a img_size[0]
    # Cálculo predefinido de hop_length
    hop_length: int = (5 * 32_000 - 2048) // (480 - 1)
    center: bool = False
    fmin: int = 500
    fmax: int = 12_500
    top_db: int = 80

    # Configuración de salida
    out_dir: str = "/content/drive/My Drive/images_spectograms_480/"
    jpeg_quality: int = 100

# Crear una instancia de la configuración
cfg = Config()
cfg.dict()

<ipython-input-22-b33f28910648>:30: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.10/migration/
  cfg.dict()


{'base_dir': '/content/drive/My Drive/',
 'train_sound_dir': '/content/drive/My Drive/songs/',
 'path_train': '/content/drive/My Drive/data_passerifromes/paths_cantos_passeriformes.csv',
 'sample_rate': 32000,
 'img_size': (480, 480),
 'seconds': 5,
 'num_offset_max': 24,
 'min_duration': 0.5,
 'n_fft': 2048,
 'n_mels': 480,
 'hop_length': 329,
 'center': False,
 'fmin': 500,
 'fmax': 12500,
 'top_db': 80,
 'out_dir': '/content/drive/My Drive/images_spectograms_480/',
 'jpeg_quality': 100}

# Prepare

In [3]:
data = pd.read_csv(cfg.path_train)
data["path_ogg"] = data["audio_path"]

In [4]:
le = LabelEncoder()
data['label_encoder'] = le.fit_transform(data['label'])

In [5]:
print(data.shape)
data.head(10)

(31373, 7)


,label,Genus,Family,audio_path,Suborder,path_ogg,label_encoder
0,Grallaria guatimalensis,Grallaria,Grallaridae,/content/drive/My Drive/songs/Grallaridae/Gral...,tiranni,/content/drive/My Drive/songs/Grallaridae/Gral...,0
1,Grallaria guatimalensis,Grallaria,Grallaridae,/content/drive/My Drive/songs/Grallaridae/Gral...,tiranni,/content/drive/My Drive/songs/Grallaridae/Gral...,0
2,Grallaria guatimalensis,Grallaria,Grallaridae,/content/drive/My Drive/songs/Grallaridae/Gral...,tiranni,/content/drive/My Drive/songs/Grallaridae/Gral...,0
3,Grallaria guatimalensis,Grallaria,Grallaridae,/content/drive/My Drive/songs/Grallaridae/Gral...,tiranni,/content/drive/My Drive/songs/Grallaridae/Gral...,0
4,Grallaria guatimalensis,Grallaria,Grallaridae,/content/drive/My Drive/songs/Grallaridae/Gral...,tiranni,/content/drive/My Drive/songs/Grallaridae/Gral...,0
5,Grallaria guatimalensis,Grallaria,Grallaridae,/content/drive/My Drive/songs/Grallaridae/Gral...,tiranni,/content/drive/My Drive/songs/Grallaridae/Gral...,0
6,Grallaria guatimalensis,Grallaria,Grallaridae,/content/drive/My Drive/songs/Grallaridae/Gral...,tiranni,/content/drive/My Drive/songs/Grallaridae/Gral...,0
7,Grallaria guatimalensis,Grallaria,Grallaridae,/content/drive/My Drive/songs/Grallaridae/Gral...,tiranni,/content/drive/My Drive/songs/Grallaridae/Gral...,0
8,Grallaria guatimalensis,Grallaria,Grallaridae,/content/drive/My Drive/songs/Grallaridae/Gral...,tiranni,/content/drive/My Drive/songs/Grallaridae/Gral...,0
9,Grallaria guatimalensis,Grallaria,Grallaridae,/content/drive/My Drive/songs/Grallaridae/Gral...,tiranni,/content/drive/My Drive/songs/Grallaridae/Gral...,0


In [6]:
def get_duration(path):
    try:
        return sf.info(path).duration
    except:
        return 0.0

def process_chunk(paths_chunk):
    return [get_duration(path) for path in paths_chunk]

In [ ]:
# Asumiendo que 'data' es el DataFrame original
paths = data["path_ogg"].tolist()
chunks = np.array_split(paths, os.cpu_count())

durations = Parallel(n_jobs=-1, verbose=1)(
    delayed(process_chunk)(chunk) for chunk in chunks
)

data["duration"] = [dur for sublist in durations for dur in sublist]

In [ ]:
data["num_offset"] = (1 + (data["duration"] - cfg.min_duration) // cfg.seconds).astype('int')
data["num_offset"] = data["num_offset"].clip(upper=cfg.num_offset_max)

In [ ]:
data.head(10)

,label,Genus,Family,audio_path,Suborder,path_ogg,label_encoder,duration,num_offset
0,Grallaria guatimalensis,Grallaria,Grallaridae,/content/drive/My Drive/songs/Grallaridae/Gral...,tiranni,/content/drive/My Drive/songs/Grallaridae/Gral...,0,63.737542,13
1,Grallaria guatimalensis,Grallaria,Grallaridae,/content/drive/My Drive/songs/Grallaridae/Gral...,tiranni,/content/drive/My Drive/songs/Grallaridae/Gral...,0,5.318503,1
2,Grallaria guatimalensis,Grallaria,Grallaridae,/content/drive/My Drive/songs/Grallaridae/Gral...,tiranni,/content/drive/My Drive/songs/Grallaridae/Gral...,0,340.279917,24
3,Grallaria guatimalensis,Grallaria,Grallaridae,/content/drive/My Drive/songs/Grallaridae/Gral...,tiranni,/content/drive/My Drive/songs/Grallaridae/Gral...,0,12.082404,3
4,Grallaria guatimalensis,Grallaria,Grallaridae,/content/drive/My Drive/songs/Grallaridae/Gral...,tiranni,/content/drive/My Drive/songs/Grallaridae/Gral...,0,68.785964,14
5,Grallaria guatimalensis,Grallaria,Grallaridae,/content/drive/My Drive/songs/Grallaridae/Gral...,tiranni,/content/drive/My Drive/songs/Grallaridae/Gral...,0,199.331500,24
6,Grallaria guatimalensis,Grallaria,Grallaridae,/content/drive/My Drive/songs/Grallaridae/Gral...,tiranni,/content/drive/My Drive/songs/Grallaridae/Gral...,0,45.112653,9
7,Grallaria guatimalensis,Grallaria,Grallaridae,/content/drive/My Drive/songs/Grallaridae/Gral...,tiranni,/content/drive/My Drive/songs/Grallaridae/Gral...,0,54.369705,11
8,Grallaria guatimalensis,Grallaria,Grallaridae,/content/drive/My Drive/songs/Grallaridae/Gral...,tiranni,/content/drive/My Drive/songs/Grallaridae/Gral...,0,6.146871,2
9,Grallaria guatimalensis,Grallaria,Grallaridae,/content/drive/My Drive/songs/Grallaridae/Gral...,tiranni,/content/drive/My Drive/songs/Grallaridae/Gral...,0,22.754195,5


In [ ]:
# guardar data
data.to_csv('/content/drive/My Drive/data_passerifromes/data_duration.csv', index=False)

In [15]:
data = pd.read_csv('/content/drive/My Drive/data_passerifromes/data_duration.csv')

/usr/local/lib/python3.11/dist-packages/librosa/feature/spectral.py:2143: UserWarning: Empty filters detected in mel frequency basis. Some channels will produce empty responses. Try increasing your sampling rate (and fmax) or reducing n_mels.
  mel_basis = filters.mel(sr=sr, n_fft=n_fft, **kwargs)


## Get spectogram image
In short we like to use 5 second interval spectograms as input images. But what should be done with the corner cases?
- There is a maximum number of offset considered for very long audio files.
- Very short files should be padded with zero to get a minimal length.

In [16]:
def get_mel_spec_db(path_ogg, offset):
    """Get dB scaled mel power spectrum"""
    required_len = cfg.seconds * cfg.sample_rate
    sig, dr = librosa.load(path=path_ogg, sr=cfg.sample_rate, offset=(offset * cfg.seconds), duration=cfg.seconds)
    sig = np.concatenate([sig, np.zeros((required_len - len(sig)), dtype=sig.dtype)])
    mel_spec = librosa.feature.melspectrogram(
        y=sig,
        hop_length=cfg.hop_length,
        sr=cfg.sample_rate,
        n_fft=cfg.n_fft,
        n_mels=cfg.n_mels,
        center=cfg.center,
        fmin=cfg.fmin,
        fmax=cfg.fmax,
    )
    mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max, top_db=cfg.top_db)
    return mel_spec_db

In [17]:
def normalize_img(img):
    """Normalize to uint8 image range"""
    assert img.ndim == 2, "unexpected dimension"
    v_min, v_max = np.min(img), np.max(img)
    return ((img - v_min) / (v_max - v_min) * 255).astype('uint8')

In [18]:
def process_record(rec):
    """Process a single record"""
    rec_dir = os.path.join(cfg.out_dir, rec.label)
    os.makedirs(rec_dir, exist_ok=True)
    stats = []
    base_stat = {"label": rec.label, "orig_filename": rec.audio_path}

    if not os.path.exists(rec.path_ogg):
        raise FileNotFoundError(f"File not found: {rec.path_ogg}")

    for offset in range(rec.num_offset):
        try:
            mel_spec_db = get_mel_spec_db(rec.path_ogg, offset=offset)
            img = normalize_img(mel_spec_db)
            fname = f"{pathlib.Path(rec.audio_path).stem}_{offset}.jpeg"
            path_img = os.path.join(rec_dir, fname)
            ret = cv2.imwrite(path_img, img, [cv2.IMWRITE_JPEG_QUALITY, cfg.jpeg_quality])
            stat = base_stat.copy()
            stat.update({
                "offset": offset,
                "ret": ret,
                "filename": "/".join(pathlib.Path(path_img).parts[-2:]),
            })
            stats.append(stat)
        except Exception as e:
            print(f"Error processing offset {offset} for {rec.audio_path}: {e}")
            continue
    return pd.DataFrame(stats)

def process_data(data):
    """Process dataframe"""
    errors = []
    l_stats = []
    for rec in data.itertuples():
        try:
            stats = process_record(rec)
            l_stats.append(stats)
        except FileNotFoundError as e:
            print(e)
            errors.append((rec.audio_path, str(e)))
        except Exception as e:
            print(f"Error reading {rec.audio_path}: {e}")
            errors.append((rec.audio_path, str(e)))
        gc.collect()
    combined_stats = pd.concat(l_stats, ignore_index=True)
    return combined_stats, errors

#### Dev

# Run all

In [ ]:
from joblib import parallel_backend

with parallel_backend("threading"):
    results = Parallel(n_jobs=os.cpu_count(), verbose=1)(
        delayed(process_data)(sub) for sub in np.array_split(data, os.cpu_count())
    )

[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
<ipython-input-17-0d0028863548>:5: RuntimeWarning: invalid value encountered in divide
  return ((img - v_min) / (v_max - v_min) * 255).astype('uint8')
<ipython-input-17-0d0028863548>:5: RuntimeWarning: invalid value encountered in cast
  return ((img - v_min) / (v_max - v_min) * 255).astype('uint8')
<ipython-input-16-a491a9c0fe44>:4: UserWarning: PySoundFile failed. Trying audioread instead.
  sig, dr = librosa.load(path=path_ogg, sr=cfg.sample_rate, offset=(offset * cfg.seconds), duration=cfg.seconds)
/usr/local/lib/python3.11/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/usr/local/lib/python3.11/dist-packages/librosa/feature/spectral.py:2143: UserWarning: Empty filters detected in mel frequency basis. Some cha

File not found: /content/drive/My Drive/songs/Thamnophilidae/Sakesphorus/Sakesphorus canadensis/Black-crestedAntshrike/516429.mp3
File not found: /content/drive/My Drive/songs/Falconidae/Daptrius/Daptrius ater/BlackCaracara/250764.mp3
File not found: /content/drive/My Drive/songs/Furnariidae/Dendrocincla/Dendrocincla fuliginosa/Plain-brownWoodcreeper/357945.mp3
File not found: /content/drive/My Drive/songs/Donacobiidae/Donacobius/Donacobius atricapilla/Black-cappedDonacobius/863918.mp3


In [ ]:
img_stats = [x for r in results for x in r[0] if isinstance(x, pd.DataFrame)]

if len(img_stats):
    img_stats = pd.concat(img_stats).reset_index(drop=True)
img_stats

[]

In [ ]:
print("Expected number of images:", data["num_offset"].sum())

Expected number of images: 269548


In [ ]:
def convert_bytes(num):
    for x in ['bytes', 'KB', 'MB', 'GB', 'TB']:
        if num < 1024.0:
            return "%3.1f %s" % (num, x)
        num /= 1024.0


bs = sum(os.stat(f).st_size for f in pathlib.Path(cfg.out_dir).glob("*/*"))
print(cfg.out_dir, convert_bytes(bs))

/content/drive/My Drive/images_spectograms/ 6.4 GB
